# 6.02 Diffusion Policy：用扩散模型生成动作块

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/06_generative_policy/6.02_diffusion_policy.ipynb)


## 为什么要学

Diffusion Policy（Chi et al., RSS 2023）把上一课的 DDPM 直接用在**以观测为条件的动作块**上，在一系列操作任务上大幅超过了 BC 和 ACT 等方法，尤其擅长多模态示教。它的设计（动作块、观测条件、滚动时域执行）被 Octo、RDT、π0 等广泛借鉴。

本课：在第 3 章的双峰障碍物任务上训练一个 Diffusion Policy，看它如何同时保留"左绕"和"右绕"两种模式。

---

## 1. 从无条件生成到条件生成

上一课生成的是 $p(\mathbf x)$。策略要生成的是 $p(\mathbf A_t\mid\mathbf O_t)$，即给定观测，生成动作块。只需要让去噪网络**额外接收观测** $\mathbf O_t$：

$$
\mathcal L=\mathbb E\Big[\big\|\boldsymbol\epsilon-\boldsymbol\epsilon_\theta(\mathbf A^k_t,\,k,\,\mathbf O_t)\big\|^2\Big]
$$

（为了不和环境时间 $t$ 混淆，Diffusion Policy 论文把扩散步记作 $k$。）

### 条件怎么注入？FiLM

Diffusion Policy 用 **FiLM**（Feature-wise Linear Modulation）把观测和扩散步编码注入到每一层：

$$
\mathbf h\leftarrow\mathbf h\odot(1+\boldsymbol\gamma(\mathbf c))+\boldsymbol\beta(\mathbf c),\qquad\mathbf c=\text{Embed}(k)+\text{Embed}(\mathbf O_t)
$$

条件向量 $\mathbf c$ 生成逐通道的缩放 $\boldsymbol\gamma$ 和平移 $\boldsymbol\beta$。这和 π0.5 的 adaRMSNorm（用时间 $t$ 生成 scale/shift/gate）是同一个思想。

### 原版设计要点（对照 `real-stanford/diffusion_policy` 与 LeRobot）

| 设计 | 原版 | 本课 |
|---|---|---|
| 去噪网络 | 1D 时序卷积 UNet + FiLM | 残差 MLP + FiLM（`ChunkDenoiser`） |
| 观测 | 最近 2 帧（图像经 ResNet18 + 空间 softmax，第 1.09 课） | 当前位置 |
| 动作 | 预测 16 步，执行 8 步 | 预测 16 步，执行 8 步 |
| 噪声表 | squaredcos_cap_v2，训练 100 步 | 余弦表 100 步 |
| 推理 | DDPM 100 步或 DDIM 10 步 | DDPM 100 步 |
| 其它 | 动作归一化到 [-1,1]、EMA | 动作本来就在 [-1,1] |

---

## 2. 实现

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from vla_ladder.envs import ObstacleReachEnv, collect_demos
from vla_ladder.models.flow import DDPM, sinusoidal_time_embedding
from vla_ladder.utils import set_seed, get_device, count_params

device = get_device()
H, EXEC = 16, 8
env = ObstacleReachEnv(num_envs=100, seed=0)
demos = collect_demos(env, n_episodes=200, seed=0)
Os, As = [], []
for d in demos:
    T_ep = len(d["actions"])
    idx = np.minimum(np.arange(T_ep)[:, None] + np.arange(H), T_ep - 1)
    Os.append(d["obs"]); As.append(d["actions"][idx])
O = torch.tensor(np.concatenate(Os)).to(device); A = torch.tensor(np.concatenate(As)).to(device)
print("训练样本:", O.shape, A.shape)

### 练习 1：FiLM 残差块与条件去噪网络

In [ ]:
class FiLMResBlock(nn.Module):
    def __init__(self, dim, cond_dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.film = nn.Linear(cond_dim, 2 * dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 2 * dim), nn.SiLU(), nn.Linear(2 * dim, dim))
        nn.init.zeros_(self.film.weight); nn.init.zeros_(self.film.bias)    # 零初始化：一开始不调制

    def forward(self, h, cond):
        # TODO: scale, shift = film(cond).chunk(2, -1)；return h + mlp(norm(h) * (1 + scale) + shift)
        raise NotImplementedError


class ChunkDenoiser(nn.Module):
    def __init__(self, obs_dim, act_dim, horizon, dim=256, depth=4, time_dim=64):
        super().__init__()
        self.horizon, self.act_dim, self.time_dim = horizon, act_dim, time_dim
        self.time_mlp = nn.Sequential(nn.Linear(time_dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.obs_mlp = nn.Sequential(nn.Linear(obs_dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.in_proj = nn.Linear(horizon * act_dim, dim)
        self.blocks = nn.ModuleList([FiLMResBlock(dim, dim) for _ in range(depth)])
        self.out = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, horizon * act_dim))

    def forward(self, x, t, obs):
        """x: (B, H, A) 带噪动作块；t: (B,) ∈ [0,1]；obs: (B, obs_dim) -> (B, H, A)"""
        B = x.shape[0]
        # TODO:
        # 1) cond = time_mlp(sinusoidal_time_embedding(t, time_dim)) + obs_mlp(obs)
        # 2) h = in_proj(x 展平)；依次过 blocks(h, cond)
        # 3) 输出 out(h) 并 view 回 (B, H, A)
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class FiLMResBlock(nn.Module):
    def __init__(self, dim, cond_dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.film = nn.Linear(cond_dim, 2 * dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 2 * dim), nn.SiLU(), nn.Linear(2 * dim, dim))
        nn.init.zeros_(self.film.weight); nn.init.zeros_(self.film.bias)

    def forward(self, h, cond):
        scale, shift = self.film(cond).chunk(2, dim=-1)
        return h + self.mlp(self.norm(h) * (1 + scale) + shift)


class ChunkDenoiser(nn.Module):
    def __init__(self, obs_dim, act_dim, horizon, dim=256, depth=4, time_dim=64):
        super().__init__()
        self.horizon, self.act_dim, self.time_dim = horizon, act_dim, time_dim
        self.time_mlp = nn.Sequential(nn.Linear(time_dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.obs_mlp = nn.Sequential(nn.Linear(obs_dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.in_proj = nn.Linear(horizon * act_dim, dim)
        self.blocks = nn.ModuleList([FiLMResBlock(dim, dim) for _ in range(depth)])
        self.out = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, horizon * act_dim))

    def forward(self, x, t, obs):
        """x: (B, H, A) 带噪动作块；t: (B,) ∈ [0,1]；obs: (B, obs_dim) -> (B, H, A)"""
        B = x.shape[0]
        cond = self.time_mlp(sinusoidal_time_embedding(t, self.time_dim)) + self.obs_mlp(obs)
        h = self.in_proj(x.reshape(B, -1))
        for blk in self.blocks:
            h = blk(h, cond)
        return self.out(h).view(B, self.horizon, self.act_dim)

In [ ]:
# ✅ 自测
net = ChunkDenoiser(2, 2, H).to(device)
out = net(torch.randn(5, H, 2, device=device), torch.rand(5, device=device), torch.randn(5, 2, device=device))
assert out.shape == (5, H, 2)
print("通过 ✅ 参数量", count_params(net))

### 训练

扩散的训练和上一课一模一样，只是网络多了一个观测输入。这里直接用配套代码包里的 `DDPM` 类（它的 `loss`、`sample` 方法就是上一课你写的 `ddpm_loss`、`p_sample_loop`）。

In [ ]:
def train_policy(loss_fn, steps=4000, lr=1e-3, seed=0):
    set_seed(seed)
    net = ChunkDenoiser(2, 2, H).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    for step in range(steps):
        i = torch.randint(0, len(O), (256,), device=device)
        loss = loss_fn(net, A[i], O[i])
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if step % 1000 == 0 or step == steps - 1:
            print(f"step {step:4d}  loss {loss.item():.4f}")
    return net.eval()

ddpm = DDPM(T=100, device=device)
dp_net = train_policy(ddpm.loss)

### 练习 2：滚动时域执行（receding horizon）

每次：观测 → 采样一个 16 步的动作块 → **执行前 8 步** → 重新观测。返回成功率、碰撞率和**左绕比例**。

In [ ]:
@torch.no_grad()
def evaluate(sample_fn, n=200, seed=7, exec_steps=EXEC):
    """sample_fn(obs (n, 2)) -> 动作块 (n, H, 2)"""
    env = ObstacleReachEnv(num_envs=n, seed=seed)
    obs = env.reset(); trajs = [obs.copy()]
    while not env.done.all():
        # TODO: chunk = sample_fn(观测张量).clamp(-1, 1)；执行前 exec_steps 步，记录轨迹
        raise NotImplementedError
    traj = np.stack(trajs, 1); succ = info["success"]
    left = (traj[:, :, 0].min(1) < -0.3)[succ].mean() if succ.any() else float("nan")
    return {"success": succ.mean(), "collide": info["collided"].mean(), "left": left}, traj

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
@torch.no_grad()
def evaluate(sample_fn, n=200, seed=7, exec_steps=EXEC):
    """sample_fn(obs (n, 2)) -> 动作块 (n, H, 2)"""
    env = ObstacleReachEnv(num_envs=n, seed=seed)
    obs = env.reset(); trajs = [obs.copy()]
    while not env.done.all():
        chunk = sample_fn(torch.tensor(obs, device=device)).clamp(-1, 1).cpu().numpy()
        for k in range(exec_steps):
            obs, _, _, info = env.step(chunk[:, k]); trajs.append(obs.copy())
    traj = np.stack(trajs, 1); succ = info["success"]
    left = (traj[:, :, 0].min(1) < -0.3)[succ].mean() if succ.any() else float("nan")
    return {"success": succ.mean(), "collide": info["collided"].mean(), "left": left}, traj

In [ ]:
# ✅ 自测
dp_sample = lambda obs: ddpm.sample(dp_net, obs, (obs.shape[0], H, 2))
r, traj = evaluate(dp_sample)
print(f"Diffusion Policy：成功率 {r['success']:.2f}，碰撞率 {r['collide']:.2f}，成功回合中左绕的比例 {r['left']:.2f}")
assert r["success"] > 0.9 and 0.15 < r["left"] < 0.85
env.plot(list(traj[:60]), title=f"Diffusion Policy: success={r['success']:.2f}, left={r['left']:.2f}"); plt.show()
print("通过 ✅ 两种模式都保留了下来，而且每条轨迹都'坚定地'走一边")

与 3.02 课的对比：

| 策略 | 成功率 | 多样性 |
|---|---|---|
| MSE 动作块 | ≈0（直冲障碍物） | 无 |
| MDN 动作块 | ≈0.9 | 有 |
| ACT（β=10，分块执行） | 不稳定 | 无（只走一边） |
| **Diffusion Policy** | **>0.95** | **有** |

---

## 3. 观察：起点处的动作分布

在起点处采样 500 个动作块，看看它们长什么样：

In [ ]:
with torch.no_grad():
    o0 = torch.tensor([[0.0, -0.8]], device=device).repeat(500, 1)
    chunks = dp_sample(o0).clamp(-1, 1).cpu().numpy()          # (500, H, 2)
paths = np.cumsum(chunks * 0.05, axis=1) + np.array([0.0, -0.8])
ax = env.plot(title="500 sampled action chunks at the start state")
for p in paths[:200]:
    ax.plot(p[:, 0], p[:, 1], "b-", alpha=0.1)
plt.show()
print("向左的比例:", (paths[:, -1, 0] < 0).mean().round(2))

两束清晰分开的轨迹：**每个采样都是一个完整、一致的模式**，中间没有"平均"的轨迹。这是扩散模型最重要的性质。

### 练习 3：执行步数的影响

把 `exec_steps` 分别设为 1、4、8、16，比较成功率与推理次数。思考：为什么执行步数太少（每步都重新采样）可能出问题？（回忆 5.02 课"采样的 z + 时间集成"。）

In [ ]:
def compare_exec_steps(steps_list=(1, 4, 8, 16)):
    """返回 {exec_steps: 成功率}"""
    # TODO
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def compare_exec_steps(steps_list=(1, 4, 8, 16)):
    """返回 {exec_steps: 成功率}"""
    out = {}
    for k in steps_list:
        r, _ = evaluate(dp_sample, n=100, exec_steps=k)
        out[k] = r["success"]
        print(f"exec_steps={k:2d}：成功率 {r['success']:.2f}，左绕比例 {r['left']:.2f}")
    return out

In [ ]:
res = compare_exec_steps()

> **读结果**：在这个任务上，即使每步都重新采样（exec_steps=1）也能成功，因为一旦往某一边走了一两步，**当前位置本身就"告诉"了模型该走哪个模式**，后续采样会自然地保持一致。但在"状态无法区分模式"的情形下（比如机器人停在原地犹豫、物体被遮挡），频繁重采样会让相邻两次选了不同模式，轨迹来回抖动；而执行太多步（16）又失去了反馈纠错的机会。所以 Diffusion Policy 论文推荐"预测 16、执行 8"的折中。π0 在真机上通常"预测 50、执行 25 左右"，并配合 RTC（实时分块）让相邻块平滑衔接。另外，每步都推理意味着推理次数是 exec_steps=8 时的 8 倍，对 30 亿参数的 VLA 来说往往不现实。

---

## 小结

- Diffusion Policy = 条件 DDPM（以观测为条件）+ 动作块 + 滚动时域执行。
- 条件通过 FiLM 注入（和 π0.5 的 adaRMSNorm 是同一思想）。
- 它能保留示教中的所有模式，每次采样都是一个"坚定"的完整模式。
- 缺点：采样要迭代很多步（DDPM 100 步），推理慢。下一课的流匹配用更"直"的路径，π0 只用 10 步。